# 🍽️ Tu primer agente con LangChain: el agente **Foodie**

**Workshop hands-on · Ciencia de datos · ≈ 2 h 50 min**

Vas a construir, paso a paso, un agente que recomienda **platos y restaurantes de la gastronomía peruana** a partir de lo que dijeron youtubers de comida en sus videos, y que además puede **verificar información actual en internet**.

| Parte | Qué haces | Tiempo |
|---|---|---|
| **0 · Entorno** | `.venv`, archivo `.env`, modelo `gpt-5-mini` | 10 min |
| **1 · Workflows vs agentes** | Definiciones (Anthropic) + un **workflow corto** que convierte una transcripción en datos | 20 min |
| **2 · Primer agente ReAct** | Ciclo Thought → Action → Observation con una herramienta local | 30 min |
| **3 · Herramientas externas con MCP** | Implementas un **servidor MCP sobre Tavily** y lo consumes desde el agente | 55 min |
| **4 · Testing y debugging** | Trayectorias, pruebas por capas, mini-evaluación y fallas sembradas | 40 min |
| **Cierre** | El agente completo, con memoria de conversación | 15 min |

> ⚠️ **Datos de demostración.** Si no existe `data/menciones.csv`, el notebook usa `data/menciones_demo.csv`: youtubers y restaurantes **ficticios** (Canal Demo A/B/C). Sirven para practicar, no para recomendar comida de verdad.

Las celdas marcadas con **`# TODO`** son las que completas tú. En la versión de solución ya están resueltas.

## Parte 0 · Entorno (≈ 10 min)

Trabajaremos con un **entorno virtual `.venv`** (así las versiones de las librerías quedan aisladas del resto de tu máquina) y con un archivo **`.env`** para las claves (así nunca quedan escritas en el notebook ni en Git).

Desde una terminal, en la carpeta del proyecto:

```bash
# 1) Crear y activar el entorno virtual
python3 -m venv .venv
source .venv/bin/activate              # Windows (PowerShell): .venv\Scripts\Activate.ps1

# 2) Instalar las dependencias (versiones fijadas)
pip install -r requirements.txt

# 3) Registrar un kernel de Jupyter que use este entorno
python -m ipykernel install --user --name foodie-lab --display-name "Python (.venv foodie-lab)"

# 4) Crear tu archivo .env a partir de la plantilla y completar tus claves
cp .env.example .env                   # luego edítalo: OPENAI_API_KEY y TAVILY_API_KEY
```

Después abre este notebook y elige el kernel **«Python (.venv foodie-lab)»**. El archivo `.env` está en `.gitignore`: **no lo subas a ningún repositorio**.

In [1]:
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path

# ¿Estamos dentro de un entorno virtual?
en_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
print(f"Python {sys.version.split()[0]} | entorno virtual: {en_venv} | {sys.prefix}")
if not en_venv:
    print("⚠️  No estás en un .venv. Cambia el kernel a «Python (.venv foodie-lab)».")

for paquete in ["langchain", "langchain-openai", "mcp", "tavily-python", "pandas", "python-dotenv"]:
    print(f"{paquete:18} {version(paquete)}")

Python 3.13.7 | entorno virtual: True | c:\LAB\2027\wrkshpd1\foodie-agent-lab\.venv
langchain          1.5.2
langchain-openai   1.8.5
mcp                2.3.0
tavily-python      0.8.5
pandas             3.1.1
python-dotenv      1.2.4


In [2]:
from dotenv import find_dotenv, load_dotenv

# Lee el archivo .env (buscándolo desde la carpeta actual hacia arriba)
load_dotenv(find_dotenv(usecwd=True), override=True)

faltan = [k for k in ("OPENAI_API_KEY", "TAVILY_API_KEY") if not os.getenv(k)]
assert not faltan, f"Faltan variables en tu archivo .env: {faltan}"
print("Claves cargadas ✔  (nunca las imprimimos)")

Claves cargadas ✔  (nunca las imprimimos)


In [3]:
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5-mini"   # cambiarlo aquí cambia el modelo de TODO el notebook

llm = ChatOpenAI(model=MODEL_NAME, reasoning_effort="low", timeout=90, max_retries=2)
print(llm.invoke("Responde solo con la palabra: listo").text)

listo


> **Sobre `gpt-5-mini`.** Es un *modelo de razonamiento*: no admite `temperature` personalizada y su cadena de pensamiento interna **no se expone**. Por eso, al observar al agente veremos sus **acciones** (llamadas a herramientas) y **observaciones** (resultados), pero no su "Thought" literal. Usamos `reasoning_effort="low"` para que responda rápido y barato.

## Parte 1 · Workflows vs agentes (≈ 20 min)

Antes de escribir un agente conviene preguntarse: **¿realmente necesito uno?** El post de Anthropic [*Building effective agents*](https://www.anthropic.com/engineering/building-effective-agents) propone una distinción arquitectónica muy útil. Ambos son **sistemas agénticos**, pero:

| | **Workflow** | **Agente** |
|---|---|---|
| **Definición** (parafraseada del post) | LLMs y herramientas se **orquestan mediante rutas de código definidas de antemano**. Tú decides el orden de los pasos. | El **propio LLM dirige su proceso y el uso de herramientas**, y conserva el control sobre *cómo* resuelve la tarea. |
| **Quién decide el siguiente paso** | Tu código | El modelo |
| **Cuándo conviene** | Tareas **bien definidas** que se pueden descomponer en subtareas fijas: buscas **predictibilidad y consistencia**. | Problemas **abiertos**, donde no puedes prever cuántos pasos hacen falta ni fijar un camino: necesitas **flexibilidad y decisiones del modelo**. |
| **Costo** | Menor latencia y costo | Mayor latencia y costo; los errores pueden acumularse, así que exige pruebas y límites (*guardrails*) |

**La recomendación central del post:** empieza por la solución **más simple posible** y aumenta la complejidad solo cuando haga falta. A veces ni siquiera necesitas un sistema agéntico: una sola llamada al LLM bien instruida (con contexto o ejemplos) puede bastar.

**Patrones de workflow** que describe el post (te recomiendo leerlo completo): *prompt chaining* (cadena de pasos donde cada llamada procesa la salida de la anterior), *routing* (clasificar la entrada y derivarla a un camino especializado), *parallelization* (subtareas en paralelo o votación), *orchestrator-workers* (un LLM central reparte subtareas) y *evaluator-optimizer* (un LLM genera y otro critica en bucle).

> ℹ️ El post incluye un aviso de que parte del panorama de herramientas cambió desde su publicación (diciembre de 2024). Las definiciones y los criterios de decisión siguen siendo la referencia.

### ¿Qué usamos en Foodie?

| Etapa del proyecto | ¿Conozco los pasos de antemano? | Enfoque |
|---|---|---|
| Convertir **una transcripción** en menciones estructuradas (restaurante, plato, distrito…) | **Sí**: extraer → validar → normalizar, siempre igual | **Workflow** (*prompt chaining* con una compuerta) |
| Responder *«¿dónde como un buen ceviche en Surquillo, y sigue abierto?»* | **No**: depende de la pregunta, de lo que haya en los datos y de si hay que verificar en la web | **Agente** |

Ahora construimos el **workflow corto**. Después de esta parte, todo el lab se enfoca en el **agente**.

### 1.1 Un workflow corto: de transcripción a datos (prompt chaining)

```
transcripción ──► [1 · LLM: extraer menciones] ──► [2 · compuerta en código] ──► [3 · normalizar] ──► DataFrame
                       (salida estructurada)        (rechaza lo dudoso)          (nombres canónicos)
```

La ruta es **fija**: el LLM no decide qué paso sigue. La **compuerta** (*gate*) del paso 2 es el control programático que el post sugiere para asegurar que el proceso no se desvíe.

In [4]:
from pydantic import BaseModel, Field


class Mencion(BaseModel):
    restaurante: str = Field(description="Nombre del restaurante o puesto")
    plato: str = Field(description="Plato probado o recomendado. Vacío si no se menciona")
    distrito: str = Field(description="Distrito donde está el lugar. Vacío si no se menciona")
    categoria: str = Field(description="ceviche, criolla, anticuchos, chifa, nikkei, postres, sandwiches, pescados y mariscos u otra")
    precio: str = Field(description="'$' económico, '$$' medio, '$$$' caro. Vacío si no hay información")
    comentario: str = Field(description="Resumen BREVE, con tus propias palabras, de la opinión del youtuber")


class Extraccion(BaseModel):
    menciones: list[Mencion]

In [5]:
import unicodedata

import pandas as pd


def normalizar(texto: str) -> str:
    """Minúsculas y sin tildes: sirve para comparar textos sin que importen mayúsculas ni acentos."""
    texto = unicodedata.normalize("NFKD", str(texto).strip().casefold())
    return "".join(c for c in texto if not unicodedata.combining(c))


DISTRITOS_VALIDOS = {
    normalizar(d): d
    for d in [
        "Lima", "Barranco", "Miraflores", "San Isidro", "Surquillo", "Santiago de Surco", "Lince",
        "Jesús María", "Magdalena del Mar", "Pueblo Libre", "San Miguel", "San Borja", "La Molina",
        "Chorrillos", "Rímac", "Breña", "Callao",
    ]
}


# ── Paso 1 · LLM con salida estructurada ─────────────────────────────────────
extractor = llm.with_structured_output(Extraccion)


def paso_extraer(texto: str) -> list[Mencion]:
    prompt = (
        "Extrae las menciones de restaurantes de esta transcripción de un youtuber de gastronomía peruana. "
        "Incluye solo lugares que el youtuber visitó o recomendó. No inventes datos: si algo no se dice, "
        "déjalo vacío. Resume el comentario con tus propias palabras (máximo 25 palabras).\n\n"
        f"TRANSCRIPCIÓN:\n{texto}"
    )
    return extractor.invoke(prompt).menciones

In [6]:
# ── Paso 2 · Compuerta (código, sin LLM) ─────────────────────────────────────
def compuerta(m: Mencion) -> tuple[bool, str]:
    """Decide si una mención es confiable. Devuelve (aceptada, motivo)."""
    if not m.restaurante.strip():
        return False, "falta el restaurante"
    if not m.plato.strip():
        return False, "falta el plato"
    if normalizar(m.distrito) not in DISTRITOS_VALIDOS:
        return False, f"distrito no válido: '{m.distrito}'"
    return True, "ok"


# ── Paso 3 · Normalización (código, sin LLM) ─────────────────────────────────
def paso_normalizar(m: Mencion, youtuber: str, video_url: str) -> dict:
    return {
        "youtuber": youtuber,
        "video_url": video_url,
        "restaurante": m.restaurante.strip(),
        "plato": m.plato.strip(),
        "distrito": DISTRITOS_VALIDOS[normalizar(m.distrito)],
        "categoria": normalizar(m.categoria) or "otra",
        "precio": m.precio.strip() if m.precio.strip() in {"$", "$$", "$$$"} else "",
        "comentario": m.comentario.strip(),
    }


# ── El workflow: una ruta FIJA, decidida por nuestro código ──────────────────
def workflow_extraer(texto: str, youtuber: str, video_url: str, verbose: bool = True):
    crudas = paso_extraer(texto)                                   # 1) LLM
    if verbose:
        print(f"1) El LLM extrajo {len(crudas)} menciones")
    aceptadas, rechazadas = [], []
    for m in crudas:                                               # 2) compuerta
        ok, motivo = compuerta(m)
        (aceptadas if ok else rechazadas).append((m, motivo))
    if verbose:
        print(f"2) La compuerta aceptó {len(aceptadas)} y rechazó {len(rechazadas)}")
        for m, motivo in rechazadas:
            print(f"   ✖ {m.restaurante or '(sin nombre)'} → {motivo}")
    filas = [paso_normalizar(m, youtuber, video_url) for m, _ in aceptadas]   # 3) normalizar
    return pd.DataFrame(filas), rechazadas

In [7]:
texto_demo = Path("data/transcripcion_demo.txt").read_text(encoding="utf-8")
print(texto_demo[:300], "...\n")

df_workflow, rechazadas = workflow_extraer(texto_demo, "Canal Demo A", "https://www.youtube.com/watch?v=DEMO0001")
df_workflow

Hola a todos, bienvenidos de nuevo al canal. Hoy nos vamos de ruta gastronómica por Lima y les prometo que no se van a arrepentir. Primera parada, Surquillo, a unas cuadras del mercado. Entramos a Marea Azul, una cevichería chiquita pero con una cola que da la vuelta a la cuadra. Pedimos el ceviche  ...

1) El LLM extrajo 3 menciones
2) La compuerta aceptó 2 y rechazó 1
   ✖ Dulce Rincón → falta el plato


,youtuber,video_url,restaurante,plato,distrito,categoria,precio,comentario
0,Canal Demo A,https://www.youtube.com/watch?v=DEMO0001,Marea Azul,ceviche clásico,Surquillo,ceviche,$$,Pescado fresquísimo y leche de tigre picante q...
1,Canal Demo A,https://www.youtube.com/watch?v=DEMO0001,Doña Lucha,anticuchos de corazón,Barranco,anticuchos,$,"Sazón tremenda, muy baratos; recomiendan visit..."


**Observa:** el workflow rechazó las menciones dudosas (un distrito que no existe, un plato que el youtuber no recordaba). Con **cualquier** transcripción recorre exactamente los mismos tres pasos. Eso es un workflow: predecible, barato y fácil de probar.

**Pero ¿qué pasa cuando el usuario pregunta algo abierto?** *«Voy con mis hijos y no tolero el picante, ¿qué me recomiendas cerca de Barranco? ¿Sigue abierto?»* No hay una secuencia fija que programar: hay que decidir qué consultar, si los datos alcanzan, si conviene ir a internet… Para eso necesitamos un **agente**. A partir de aquí, **todo el lab se enfoca en construir ese agente hasta dejarlo completo**.

> 📎 *Anexo A (al final):* cómo aplicar este mismo workflow a muchas transcripciones reales para generar la tabla `data/menciones.csv`.

## Parte 2 · Tu primer agente ReAct (≈ 30 min)

**ReAct** (*Reason + Act*, Yao et al., 2022) es el bucle que hay detrás de casi todo agente con herramientas:

```
        ┌──────────────────────────────────────────────┐
        ▼                                              │
  pregunta ─► 🧠 el LLM razona y decide ─► 🔧 Acción (llamar a una herramienta)
                         │                              │
                         │                      👀 Observación (resultado)
                         ▼                              │
                 ✅ respuesta final  ◄──────────────────┘   (cuando ya tiene lo necesario)
```

La diferencia con un workflow es **quién decide**: aquí el modelo elige si llamar una herramienta, cuál, con qué argumentos, y cuándo parar. En LangChain 1.x, `create_agent` construye este bucle (sobre LangGraph) con dos nodos: **`model`** y **`tools`**.

Empezamos con **una sola herramienta local**: una consulta sobre la tabla de menciones de los youtubers.

In [8]:
DATA_DIR = Path("data")
MENCIONES_CSV = DATA_DIR / "menciones.csv"
if not MENCIONES_CSV.exists():
    MENCIONES_CSV = DATA_DIR / "menciones_demo.csv"
    print("⚠️  Usando datos DEMO ficticios (data/menciones_demo.csv).")

df_menciones = pd.read_csv(MENCIONES_CSV)
print(f"{len(df_menciones)} menciones | {df_menciones['restaurante'].nunique()} restaurantes | "
      f"{df_menciones['distrito'].nunique()} distritos | {df_menciones['youtuber'].nunique()} youtubers")
df_menciones.groupby(["distrito", "categoria"]).size().unstack(fill_value=0)

⚠️  Usando datos DEMO ficticios (data/menciones_demo.csv).
16 menciones | 12 restaurantes | 8 distritos | 3 youtubers


categoria,anticuchos,ceviche,chifa,criolla,nikkei,pescados y mariscos,postres,sandwiches
distrito,,,,,,,,
Barranco,1,0,0,1,0,0,1,0
Chorrillos,0,1,0,0,0,1,0,0
Lima,0,0,0,1,0,0,0,0
Lince,0,0,2,0,0,0,0,0
Miraflores,0,0,0,0,2,0,0,0
San Isidro,0,0,0,1,0,0,0,0
Santiago de Surco,1,0,0,0,0,0,0,0
Surquillo,0,3,0,0,0,0,0,1


### 2.1 La herramienta `consultar_menciones`

Una herramienta es **una función Python + una descripción**. El LLM nunca ve tu código: solo lee el **nombre, la descripción y el esquema de argumentos**. Por eso el *docstring* es, en la práctica, un prompt.

In [9]:
from langchain_core.tools import tool


@tool(parse_docstring=True)
def consultar_menciones(
    distrito: str = "",
    plato_o_categoria: str = "",
    texto: str = "",
    precio_maximo: str = "",
    limite: int = 5,
) -> str:
    """Consulta lo que dijeron youtubers de gastronomía peruana en sus videos (la base de conocimiento de Foodie).

    Úsala SIEMPRE antes de recomendar un plato o un restaurante. Todos los filtros son opcionales y se combinan.

    Args:
        distrito: Distrito a consultar, por ejemplo "Barranco" o "Surquillo".
        plato_o_categoria: Plato o categoría, por ejemplo "ceviche", "anticuchos", "chifa", "postres".
        texto: UNA palabra clave para buscar en los comentarios, por ejemplo "picante" o "niños".
        precio_maximo: Nivel de precio máximo: "$" (económico), "$$" (medio) o "$$$" (caro).
        limite: Cantidad máxima de resultados (entre 1 y 10).
    """
    res = df_menciones
    if distrito:
        res = res[res["distrito"].map(normalizar) == normalizar(distrito)]
    if plato_o_categoria:
        q = normalizar(plato_o_categoria)
        res = res[res["plato"].map(normalizar).str.contains(q, regex=False)
                  | res["categoria"].map(normalizar).str.contains(q, regex=False)]
    if texto:
        res = res[res["comentario"].map(normalizar).str.contains(normalizar(texto), regex=False)]
    if precio_maximo:
        niveles = {"$": 1, "$$": 2, "$$$": 3}
        res = res[res["precio"].map(niveles).fillna(3) <= niveles.get(precio_maximo, 3)]

    if res.empty:
        return "SIN_RESULTADOS: ningún youtuber de la base habló de eso con esos filtros."

    limite = max(1, min(limite, 10))
    lineas = []
    for i, fila in enumerate(res.head(limite).itertuples(), 1):
        lineas.append(
            f"[{i}] {fila.restaurante} — {fila.plato} ({fila.distrito}, precio {fila.precio or 'n/d'})\n"
            f"    Fuente: {fila.youtuber} · {fila.video_url}\n"
            f"    Comentario: {fila.comentario}"
        )
    return "\n".join(lineas)

**Primera regla de oro del testing de agentes:** prueba la herramienta **sola**, sin LLM, antes de dársela al agente.

In [10]:
print(consultar_menciones.invoke({"distrito": "surquillo", "plato_o_categoria": "ceviche"}))
print("-" * 60)
print(consultar_menciones.invoke({"distrito": "Jesús María"}))   # no hay datos: ¿qué devuelve?

[1] Cevichería Marea Azul — Ceviche clásico (Surquillo, precio $$)
    Fuente: Canal Demo A · https://www.youtube.com/watch?v=DEMO0001
    Comentario: Pescado muy fresco y leche de tigre potente y picante; se llena al mediodía.
[2] Cevichería Marea Azul — Tiradito al ají amarillo (Surquillo, precio $$)
    Fuente: Canal Demo B · https://www.youtube.com/watch?v=DEMO0003
    Comentario: Corte fino y salsa cremosa; menos picante que el ceviche.
[3] Cevichería Marea Azul — Ceviche de conchas negras (Surquillo, precio $$$)
    Fuente: Canal Demo B · https://www.youtube.com/watch?v=DEMO0008
    Comentario: Opción de lujo del local con sabor intenso a mar; el picante es a elección.
------------------------------------------------------------
SIN_RESULTADOS: ningún youtuber de la base habló de eso con esos filtros.


### 2.2 El system prompt

El *system prompt* fija el rol y las reglas. Fíjate que las reglas **no** son decoración: la regla 1 obliga a **fundamentar** cada recomendación en la herramienta, y la regla 3 define qué hacer cuando no hay datos.

In [11]:
SYSTEM_PROMPT = """Eres Foodie, un asistente que recomienda comida peruana basándose en lo que dijeron youtubers de gastronomía en sus videos.

Reglas:
1. Antes de recomendar, consulta SIEMPRE la herramienta consultar_menciones. No recomiendes nada que no salga de sus resultados.
2. Cada recomendación debe citar al youtuber y el enlace del video.
3. Si la herramienta devuelve SIN_RESULTADOS, dilo con claridad y no inventes nada; ofrece probar otro distrito o categoría.
4. Responde en español, de forma breve (máximo 3 recomendaciones) y menciona el nivel de precio cuando exista."""

### 2.3 Ver el ciclo ReAct en vivo

La función `ask` ejecuta al agente con **streaming** y muestra cada paso: la **acción** (llamada a herramienta), la **observación** (lo que devolvió) y la **respuesta final**. La usaremos en todo el lab. Es `async` porque más adelante las herramientas MCP lo son; en Jupyter puedes escribir `await` directamente en una celda.

In [12]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage


def _vista(texto, n=350) -> str:
    texto = " ".join(str(texto).split())
    return texto if len(texto) <= n else texto[:n] + "…"


async def ask(agent, pregunta: str, *, config: dict | None = None, verbose: bool = True):
    """Ejecuta el agente mostrando cada paso del ciclo ReAct. Devuelve (respuesta_final, mensajes_nuevos)."""
    mensajes, respuesta = [], ""
    if verbose:
        print(f"🧑 {pregunta}\n")
    entrada = {"messages": [{"role": "user", "content": pregunta}]}
    async for chunk in agent.astream(entrada, config=config, stream_mode="updates"):
        for _nodo, update in chunk.items():
            for msg in (update or {}).get("messages", []):
                mensajes.append(msg)
                if isinstance(msg, AIMessage) and msg.tool_calls:
                    for tc in msg.tool_calls:
                        if verbose:
                            print(f"🔧 Acción      → {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
                elif isinstance(msg, ToolMessage):
                    if verbose:
                        print(f"👀 Observación → {_vista(msg.content)}\n")
                elif isinstance(msg, AIMessage):
                    respuesta = str(msg.text)
                    if verbose:
                        print(f"🤖 Respuesta final:\n{respuesta}\n")
    return respuesta, mensajes

In [13]:
agent_v1 = create_agent(llm, tools=[consultar_menciones], system_prompt=SYSTEM_PROMPT)

In [14]:
respuesta, mensajes = await ask(agent_v1, "Quiero un ceviche bien picante en Surquillo. ¿Qué recomiendan los youtubers?")

🧑 Quiero un ceviche bien picante en Surquillo. ¿Qué recomiendan los youtubers?

🔧 Acción      → consultar_menciones({"distrito": "Surquillo", "plato_o_categoria": "ceviche", "texto": "picante", "limite": 5})
👀 Observación → [1] Cevichería Marea Azul — Ceviche clásico (Surquillo, precio $$) Fuente: Canal Demo A · https://www.youtube.com/watch?v=DEMO0001 Comentario: Pescado muy fresco y leche de tigre potente y picante; se llena al mediodía. [2] Cevichería Marea Azul — Tiradito al ají amarillo (Surquillo, precio $$) Fuente: Canal Demo B · https://www.youtube.com/watch?v…

🤖 Respuesta final:
Recomendación principal:
- Cevichería Marea Azul — Ceviche clásico (Surquillo, precio $$). Youtuber: Canal Demo A — https://www.youtube.com/watch?v=DEMO0001. Comentario: “leche de tigre potente y picante” — recomendado para un ceviche bien picante.

Opción adicional (si quieres ajustar intensidad):
- Cevichería Marea Azul — Ceviche de conchas negras (Surquillo, precio $$$). Youtuber: Canal Demo B — ht

In [15]:
respuesta, mensajes = await ask(agent_v1, "Voy con mis hijos pequeños y no tolero el picante. Algo económico, ¿qué me recomiendas?")

🧑 Voy con mis hijos pequeños y no tolero el picante. Algo económico, ¿qué me recomiendas?

🔧 Acción      → consultar_menciones({"texto": "niños", "precio_maximo": "$", "limite": 5})
👀 Observación → [1] Chifa Dragón Dorado — Arroz chaufa especial (Lince, precio $) Fuente: Canal Demo A · https://www.youtube.com/watch?v=DEMO0002 Comentario: Porciones grandes para compartir y buen precio; apto para familias con niños.

🤖 Respuesta final:
Te recomiendo:

- Arroz chaufa especial en Chifa Dragón Dorado (Lince) — porciones grandes, apto para familias con niños; precio: $. Fuente: Canal Demo A — https://www.youtube.com/watch?v=DEMO0002

Consejo: pide que lo preparen sin picante. ¿Quieres que busque otras opciones económicas en otro distrito?



### 2.4 ¿Importa el system prompt? Compruébalo

Ahora el **mismo modelo y la misma herramienta, pero sin system prompt**. Pregúntale por un distrito donde la base **no tiene datos**.

In [16]:
agent_sin_prompt = create_agent(llm, tools=[consultar_menciones])
_ = await ask(agent_sin_prompt, "Recomiéndame un pollo a la brasa en Jesús María")
_ = await ask(agent_v1, "Recomiéndame un pollo a la brasa en Jesús María")

🧑 Recomiéndame un pollo a la brasa en Jesús María

🔧 Acción      → consultar_menciones({"distrito": "Jesús María", "plato_o_categoria": "pollo a la brasa", "limite": 5})
👀 Observación → SIN_RESULTADOS: ningún youtuber de la base habló de eso con esos filtros.

🤖 Respuesta final:
He revisado la base de videos de Foodie y no hay menciones sobre “pollo a la brasa” en Jesús María. Puedo ayudarte de varias maneras — dime cuál prefieres:

Opciones
- ¿Quieres que busque recomendaciones en distritos cercanos (Lince, Magdalena, San Isidro, Pueblo Libre)? Puedo listar lugares populares ahí.
- ¿Prefieres cadenas conocidas (para delivery) o un local tradicional para comer ahí?
- ¿Cuál es tu rango de precio? ($ económico, $$ medio, $$$ caro).
- ¿Pides para llevar/Delivery o quieres ir a comer en el local?

Mientras me respondes, te dejo puntos útiles para elegir buen pollo a la brasa:
- Piel bien dorada y crujiente, carne jugosa y no reseca.
- Acompañamientos: papas fritas o sancochadas, ensalada f

> Las respuestas de un LLM **no son deterministas**: puede que el agente sin prompt acierte y que otro día falle. Por eso más adelante (Parte 4) **medimos** con varios casos en lugar de fiarnos de una sola ejecución.

**✍️ Pruébalo tú:** pídele *«anticuchos en Barranco»*, luego *«algo nikkei para una ocasión especial»*, y observa qué **argumentos** elige el agente para `consultar_menciones`. ¿Coinciden con lo que tú habrías puesto?

## Parte 3 · Herramientas externas con MCP (≈ 55 min)

Nuestra base de youtubers tiene datos **de ayer**; un restaurante puede haber cerrado. El agente necesita **la web**. En lugar de pegar Tavily directamente al agente, vamos a **construir un servidor MCP** y a consumirlo como cliente.

### ¿Qué es MCP?

**Model Context Protocol** es un protocolo abierto (creado por Anthropic) para que una aplicación de IA se conecte a **herramientas y datos** de forma estándar. Un **servidor MCP** publica herramientas (nombre + descripción + esquema JSON de argumentos); un **cliente MCP** las lista y las invoca. Lo bueno: la misma herramienta sirve a cualquier cliente compatible (tu agente, un IDE, una app de escritorio…).

```
Agente (LangChain) ─► puente ─► Cliente MCP ──stdio──► Servidor MCP (lo escribes tú) ─► API de Tavily
                                 (SDK `mcp`)            (SDK `mcp`)
```

Usamos el **SDK oficial** (`pip install mcp`). Ojo: la versión 2.x renombró `FastMCP` a **`MCPServer`**; si ves tutoriales antiguos con `FastMCP`, usan la v1.

**Transporte `stdio`:** el cliente lanza el servidor como un **subproceso** y se hablan por la entrada/salida estándar. No hay puertos ni procesos aparte que administrar.

### 3A · Implementar el servidor MCP sobre Tavily (≈ 25 min)

El servidor expone **dos herramientas**:

| Herramienta | Qué hace | Cuándo usarla |
|---|---|---|
| `web_search` | Busca en internet y devuelve títulos, URLs y fragmentos | **Descubrir** información actual |
| `extract_page` | Descarga el texto completo de **una URL** | **Leer** una página que ya conoces |

Dos detalles de diseño importantes en el código:
- **Las descripciones (docstrings)** son lo que el LLM lee para decidir cuál usar.
- **Los errores** se lanzan con `ToolError` y un mensaje claro: así el modelo recibe un motivo legible en vez de un genérico *«Error executing tool»*.
- **stdout está reservado al protocolo**: el servidor registra sus mensajes con `logging` hacia **stderr**.

In [17]:
%%writefile tavily_mcp_server.py
"""MCP Server sobre Tavily: expone web_search y extract_page por stdio."""
import logging
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from tavily import TavilyClient
from tavily.errors import InvalidAPIKeyError, UsageLimitExceededError

# Las claves viven en .env (junto a este archivo); si el cliente ya pasó una variable, esa gana.
load_dotenv(Path(__file__).parent / ".env")

# IMPORTANTE: en stdio, stdout es el canal del protocolo. Los logs SIEMPRE van a stderr.
logging.basicConfig(stream=sys.stderr, level=logging.INFO, format="[tavily-mcp] %(levelname)s %(message)s")
log = logging.getLogger("tavily-mcp")

mcp = MCPServer("tavily-foodie")


def _client() -> TavilyClient:
    key = os.getenv("TAVILY_API_KEY")
    if not key:
        raise ToolError("TAVILY_API_KEY no está definida en el servidor (revisa el archivo .env).")
    return TavilyClient(api_key=key)


def _explain(exc: Exception) -> ToolError:
    """Traduce errores de Tavily a mensajes que el MODELO pueda leer y entender.
    (Si dejamos escapar una excepción cualquiera, el modelo solo vería 'Error executing tool <nombre>'.)"""
    if isinstance(exc, InvalidAPIKeyError):
        return ToolError("Tavily rechazó la clave de API (inválida). No se puede buscar en la web ahora.")
    if isinstance(exc, UsageLimitExceededError):
        return ToolError("Se agotó la cuota de Tavily. No se puede buscar en la web ahora.")
    return ToolError(f"Falló la llamada a Tavily: {type(exc).__name__}: {exc}")


@mcp.tool()
def web_search(query: str, max_results: int = 5) -> str:
    """Busca en internet y devuelve títulos, URLs y fragmentos de texto.
    Úsala para DESCUBRIR información actual: si un restaurante sigue abierto, su dirección,
    horarios, reseñas recientes. NO sirve para leer una página completa (para eso, extract_page)."""
    max_results = max(1, min(max_results, 8))
    log.info("web_search query=%r max_results=%s", query, max_results)
    client = _client()
    try:
        data = client.search(query=query, max_results=max_results, search_depth="basic")
    except Exception as exc:
        log.error("web_search falló: %r", exc)
        raise _explain(exc) from exc
    items = data.get("results", [])
    if not items:
        return "SIN_RESULTADOS: la búsqueda no devolvió páginas."
    return "\n\n".join(
        f"[{i}] {r['title']}\nURL: {r['url']}\n{r['content'][:500]}" for i, r in enumerate(items, 1)
    )


@mcp.tool()
def extract_page(url: str) -> str:
    """Descarga y devuelve el texto completo de UNA página web a partir de su URL.
    Úsala solo cuando ya tengas una URL concreta (por ejemplo, devuelta por web_search)
    y necesites leer el detalle: la carta, el horario, una reseña larga."""
    log.info("extract_page url=%s", url)
    client = _client()
    try:
        data = client.extract(urls=[url])
    except Exception as exc:
        log.error("extract_page falló: %r", exc)
        raise _explain(exc) from exc
    results = data.get("results", [])
    if not results:
        raise ToolError(f"No se pudo extraer el contenido de {url}: {data.get('failed_results', [])}")
    return results[0]["raw_content"][:6000]


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing tavily_mcp_server.py


### Probar el servidor **sin agente**

Antes de conectarlo a un LLM, lo usamos como lo haría cualquier cliente MCP: lo lanzamos, **listamos** sus herramientas y **llamamos** a una. Si algo falla aquí, el problema es del servidor (o de Tavily), no del agente.

In [18]:
import asyncio

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_SCRIPT = str(Path("tavily_mcp_server.py").resolve())
SERVER_LOG = Path("mcp_server.log")
_server_log = open(SERVER_LOG, "a", buffering=1, encoding="utf-8")   # aquí caen los logs (stderr) del servidor


def server_params(script: str = SERVER_SCRIPT, env: dict | None = None) -> StdioServerParameters:
    """Cómo lanzar el servidor: un subproceso con ESTE mismo Python (el del .venv).
    El servidor lee su propio .env; si pasas `env`, esas variables tienen prioridad."""
    return StdioServerParameters(command=sys.executable, args=[script], env=env)


def connect(params: StdioServerParameters) -> Client:
    """Abre una conexión stdio NUEVA (lanza el subproceso). Se usa con `async with`."""
    return Client(stdio_client(params, errlog=_server_log))

In [19]:
async with connect(server_params()) as client:
    herramientas = (await client.list_tools()).tools
    for t in herramientas:
        print(f"• {t.name}")
        print(f"  descripción: {_vista(t.description, 160)}")
        print(f"  esquema    : {json.dumps(t.input_schema['properties'], ensure_ascii=False)}\n")

    resultado = await client.call_tool("web_search", {"query": "ceviche peruano origen historia", "max_results": 2})
    print("¿error?", resultado.is_error)
    print(_vista(resultado.content[0].text, 600))

• web_search
  descripción: Busca en internet y devuelve títulos, URLs y fragmentos de texto. Úsala para DESCUBRIR información actual: si un restaurante sigue abierto, su dirección, horari…
  esquema    : {"query": {"title": "Query", "type": "string"}, "max_results": {"default": 5, "title": "Max Results", "type": "integer"}}

• extract_page
  descripción: Descarga y devuelve el texto completo de UNA página web a partir de su URL. Úsala solo cuando ya tengas una URL concreta (por ejemplo, devuelta por web_search) …
  esquema    : {"url": {"title": "Url", "type": "string"}}

¿error? True
Error executing tool web_search: Tavily rechazó la clave de API (inválida). No se puede buscar en la web ahora.


In [20]:
# Los mensajes del servidor (stderr) quedaron en este archivo: úsalo para depurar
print(SERVER_LOG.read_text(encoding="utf-8")[-500:])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf3 in position 118: invalid continuation byte

### 3B · Consumir el servidor desde el agente (≈ 30 min)

El agente de LangChain espera **herramientas de LangChain**. El servidor MCP publica **herramientas MCP**. Falta un **puente**: por cada herramienta del servidor creamos una `StructuredTool` que reutiliza su **nombre**, **descripción** y **esquema JSON**, y cuyo cuerpo hace `call_tool`. Es poco código y deja ver que *una herramienta MCP es una función con un esquema*.

Decisiones del puente:
- Cada llamada abre y cierra su conexión (más simple y robusto en notebooks que mantener una sesión viva).
- Un **error de la herramienta** no rompe el agente: se devuelve como **observación** (`ERROR de la herramienta …`) para que el modelo pueda reaccionar.
- Hay un **timeout** de 60 s por llamada.

In [ ]:
from langchain_core.tools import StructuredTool


def resultado_a_texto(result) -> str:
    """Un resultado MCP trae bloques de contenido; nos quedamos con el texto."""
    return "\n".join(getattr(bloque, "text", str(bloque)) for bloque in result.content)


def mcp_a_langchain(params: StdioServerParameters, spec) -> StructuredTool:
    async def _llamar(**kwargs) -> str:
        async with connect(params) as client:
            result = await asyncio.wait_for(client.call_tool(spec.name, kwargs), timeout=60)
        texto = resultado_a_texto(result)
        return f"ERROR de la herramienta {spec.name}: {texto}" if result.is_error else texto

    return StructuredTool(
        name=spec.name,
        description=spec.description or "",
        args_schema=spec.input_schema,   # el esquema JSON que publica el servidor
        coroutine=_llamar,
    )


async def load_mcp_tools(params: StdioServerParameters) -> list[StructuredTool]:
    async with connect(params) as client:
        especificaciones = (await client.list_tools()).tools
    return [mcp_a_langchain(params, s) for s in especificaciones]

In [ ]:
mcp_tools = await load_mcp_tools(server_params())
for t in mcp_tools:
    print(t.name, "→", list(t.args))

Ahora el **system prompt v2**: añade cuándo usar cada herramienta web y cómo distinguir las fuentes.

In [ ]:
SYSTEM_PROMPT_V2 = """Eres Foodie, un asistente que recomienda comida peruana basándose en lo que dijeron youtubers de gastronomía en sus videos, y que puede verificar datos actuales en internet.

Reglas:
1. Antes de recomendar, consulta SIEMPRE consultar_menciones. No recomiendes nada que no salga de sus resultados.
2. Usa web_search SOLO para verificar datos actuales de un lugar (si sigue abierto, dirección, horario, reseñas recientes) cuando el usuario lo pida o la duda sea relevante.
3. Usa extract_page SOLO si ya tienes una URL concreta (la dio el usuario o la devolvió web_search) y necesitas leer su detalle.
4. Cita al youtuber y el enlace del video para lo que venga de consultar_menciones, y la URL para lo que venga de la web. Distingue claramente ambas fuentes.
5. Si consultar_menciones devuelve SIN_RESULTADOS, dilo con claridad y no inventes nada.
6. Si una herramienta devuelve un ERROR, cuéntalo en una frase y no inventes lo que no pudiste verificar.
7. Responde en español, de forma breve (máximo 3 recomendaciones)."""

In [ ]:
tools_v2 = [consultar_menciones] + mcp_tools
agent_v2 = create_agent(llm, tools=tools_v2, system_prompt=SYSTEM_PROMPT_V2)

Tres preguntas, **tres rutas distintas**. Nadie programó cuál seguir: el agente decide.

In [ ]:
# Ruta A: solo la base de youtubers
respuesta, mensajes = await ask(agent_v2, "Quiero un ceviche bien picante en Surquillo. ¿Qué recomiendan los youtubers?")

In [ ]:
# Ruta B: base de youtubers + verificación en la web
respuesta, mensajes = await ask(agent_v2, "¿Sigue abierta la Cevichería El Muelle Chico de Chorrillos? Verifícalo en internet y dime horario si lo encuentras.")

In [ ]:
# Ruta C: leer una página concreta
respuesta, mensajes = await ask(agent_v2, "Lee https://es.wikipedia.org/wiki/Ceviche y resúmeme en 2 líneas qué hace especial al ceviche peruano.")

> 💡 Los restaurantes de la base **demo** son inventados: es esperable que la web no los encuentre. Observa cómo se comporta el agente cuando la verificación **no confirma** lo que dijo el youtuber. Con datos reales, este contraste *«lo que dijo el youtuber» vs. «lo que dice la web hoy»* es justamente el valor del agente.

## Parte 4 · Testing y debugging (≈ 40 min)

Un agente falla de formas que un programa normal no: elige mal una herramienta, inventa datos, entra en bucles, o la herramienta falla por fuera y nadie se entera. La estrategia: **mirar la trayectoria, probar por capas y medir con casos**.

### 4.1 Inspeccionar la trayectoria

La **respuesta** puede verse bien aunque el **camino** haya sido malo (herramienta equivocada, pasos de más). Extraemos qué herramientas llamó el agente y con qué argumentos.

In [ ]:
def trayectoria(mensajes) -> list[tuple[str, dict]]:
    """Herramientas que llamó el agente, en orden: [(nombre, argumentos), ...]."""
    return [(tc["name"], tc["args"]) for m in mensajes if isinstance(m, AIMessage) for tc in m.tool_calls]


respuesta, mensajes = await ask(agent_v2, "¿Qué anticuchos recomiendan en Barranco?", verbose=False)
for paso, (nombre, args) in enumerate(trayectoria(mensajes), 1):
    print(f"paso {paso}: {nombre} {args}")
print("\nRespuesta:", _vista(respuesta, 300))

### 4.2 Pruebas por capas

Cuando algo falla, ¿qué capa tiene el problema? Se prueba de **abajo hacia arriba**:

```
capa 0  Tavily directo  →  capa 1  herramienta local  →  capa 2  servidor MCP (call_tool)  →  capa 3  agente
```

In [ ]:
from tavily import TavilyClient


async def call_mcp(nombre: str, args: dict, params: StdioServerParameters | None = None):
    async with connect(params or server_params()) as client:
        return await client.call_tool(nombre, args)


# Capa 0 · Tavily directo (¿la clave y la red funcionan?)
def test_capa0_tavily_directo():
    r = TavilyClient(api_key=os.environ["TAVILY_API_KEY"]).search("ceviche peruano", max_results=1)
    assert r["results"], "Tavily no devolvió resultados"


# Capa 1 · herramienta local
def test_capa1_local_con_datos():
    out = consultar_menciones.invoke({"distrito": "Surquillo", "plato_o_categoria": "ceviche"})
    assert "Marea Azul" in out and "Fuente:" in out


def test_capa1_local_sin_datos():
    assert consultar_menciones.invoke({"distrito": "Jesús María"}).startswith("SIN_RESULTADOS")


# Capa 2 · servidor MCP
async def test_capa2_mcp_expone_dos_tools():
    async with connect(server_params()) as client:
        nombres = {t.name for t in (await client.list_tools()).tools}
    assert nombres == {"web_search", "extract_page"}, nombres


async def test_capa2_mcp_web_search():
    r = await call_mcp("web_search", {"query": "ceviche peruano origen", "max_results": 2})
    assert not r.is_error and "URL:" in resultado_a_texto(r)


async def correr(pruebas):
    for prueba in pruebas:
        try:
            r = prueba()
            if asyncio.iscoroutine(r):
                await r
            print("✅", prueba.__name__)
        except Exception as exc:
            print("❌", prueba.__name__, "→", type(exc).__name__, exc)


await correr([test_capa0_tavily_directo, test_capa1_local_con_datos, test_capa1_local_sin_datos,
              test_capa2_mcp_expone_dos_tools, test_capa2_mcp_web_search])

### 4.3 Mini-evaluación: medir en vez de adivinar

Una sola ejecución no prueba nada con un LLM. Definimos **casos** con lo que esperamos del **comportamiento** (qué herramientas debe usar, qué no, si cita fuentes, si admite que no sabe) y los corremos todos.

In [ ]:
CASOS = [
    {"id": "ceviche-local", "pregunta": "Quiero un ceviche en Surquillo, ¿qué me recomiendan?",
     "debe_usar": {"consultar_menciones"}, "no_debe_usar": {"extract_page"}, "debe_citar_fuente": True},
    {"id": "verificar-web", "pregunta": "¿Sigue abierta la Cevichería El Muelle Chico de Chorrillos? Verifícalo en internet.",
     "debe_usar": {"consultar_menciones", "web_search"}},
    {"id": "sin-datos", "pregunta": "Recomiéndame un pollo a la brasa en Jesús María",
     "debe_usar": {"consultar_menciones"}, "debe_admitir_falta_de_datos": True},
    {"id": "leer-url", "pregunta": "Lee https://es.wikipedia.org/wiki/Ceviche y resume en 2 líneas el origen del plato.",
     "debe_usar": {"extract_page"}},
    {"id": "ninos-suave", "pregunta": "Voy con mis hijos pequeños, busco algo suave y económico.",
     "debe_usar": {"consultar_menciones"}, "debe_citar_fuente": True},
]

In [ ]:
# TODO: agrega a CASOS un caso propio (id, pregunta, debe_usar y, si aplica, debe_citar_fuente / no_debe_usar).
CASOS.append({"id": "postre-barranco", "pregunta": "¿Qué postre me recomiendan en Barranco?",
              "debe_usar": {"consultar_menciones"}, "debe_citar_fuente": True})

In [ ]:
FRASES_SIN_DATOS = ("no tengo", "no encontr", "sin resultados", "no hay", "no cuento", "no hallé", "no aparece", "no existen")


def evaluar(caso: dict, respuesta: str, mensajes: list) -> dict:
    usadas = {nombre for nombre, _ in trayectoria(mensajes)}
    checks = {
        "usa herramientas esperadas": caso.get("debe_usar", set()) <= usadas,
        "evita herramientas prohibidas": not (caso.get("no_debe_usar", set()) & usadas),
    }
    if caso.get("debe_citar_fuente"):
        citas_youtuber = any(y in respuesta for y in df_menciones["youtuber"].unique())
        checks["cita la fuente"] = citas_youtuber or "youtube.com" in respuesta
    if caso.get("debe_admitir_falta_de_datos"):
        checks["admite que no hay datos"] = any(f in respuesta.lower() for f in FRASES_SIN_DATOS)
    return checks


async def correr_evaluacion(agent, casos) -> pd.DataFrame:
    filas = []
    for caso in casos:
        try:
            respuesta, mensajes = await ask(agent, caso["pregunta"], verbose=False)
            fila = {"caso": caso["id"], **evaluar(caso, respuesta, mensajes),
                    "herramientas": [n for n, _ in trayectoria(mensajes)]}
        except Exception as exc:
            fila = {"caso": caso["id"], "error": f"{type(exc).__name__}: {exc}"}
        filas.append(fila)
    return pd.DataFrame(filas).set_index("caso")


resultados = await correr_evaluacion(agent_v2, CASOS)
resultados

> ⚠️ **Límites de esta evaluación.** Los chequeos son deliberadamente simples (herramientas usadas, nombre del youtuber en la respuesta, frases como *«no tengo»*): sirven para detectar regresiones, no para juzgar calidad. El siguiente paso serio es un **LLM como juez** con una rúbrica, y repetir cada caso varias veces para medir variabilidad.

### 4.4 Fallas sembradas: depura en grupo

Cinco fallas típicas. Para cada una: **provócala, observa los síntomas, encuentra la causa, corrígela**.

#### Falla 1 · Un `print()` en el servidor stdio

En `stdio`, **stdout es el canal del protocolo**: todo lo que el servidor escriba ahí debe ser un mensaje MCP válido. Un `print()` «inocente» lo contamina. Generamos una copia rota del servidor con un `print` y la conectamos.

In [ ]:
codigo = Path("tavily_mcp_server.py").read_text(encoding="utf-8")
roto = codigo.replace('mcp = MCPServer("tavily-foodie")', 'print("Servidor Tavily iniciando...")   # <- BUG\nmcp = MCPServer("tavily-foodie")')
assert roto != codigo, "no se pudo inyectar el bug"
ruta_rota = Path("tavily_mcp_server_roto.py")
ruta_rota.write_text(roto, encoding="utf-8")

async with connect(server_params(str(ruta_rota.resolve()))) as client:
    print("Herramientas:", [t.name for t in (await client.list_tools()).tools])

**Síntomas:** con el SDK 2.x la conexión *sobrevive*, pero el cliente imprime un traceback **«Failed to parse JSONRPC message from server»** por la línea que no es JSON. No siempre será tan benigno: una salida parcial o parecida a JSON puede **corromper mensajes** reales.

**Causa:** el texto fue a **stdout**. **Corrección:** en servidores stdio, nada de `print()`: usa `logging` hacia **stderr** (como hace `tavily_mcp_server.py`) y mira `mcp_server.log`.

In [ ]:
ruta_rota.unlink(missing_ok=True)   # limpiamos la copia rota

#### Falla 2 · Descripciones de herramienta ambiguas

El modelo decide **solo** leyendo nombre y descripción. Si degradamos las descripciones, pierde la pista de **cuándo** usar cada herramienta. Comparamos con la trayectoria.

In [ ]:
tools_vagas = [t.model_copy(update={"description": "Herramienta de internet."}) for t in mcp_tools]
agent_vago = create_agent(
    llm,
    tools=[consultar_menciones] + tools_vagas,
    system_prompt="Eres Foodie. Ayuda con recomendaciones de comida peruana.",
)

pregunta = "Lee https://es.wikipedia.org/wiki/Ceviche y dime en una línea qué es el leche de tigre."
_, msgs_vago = await ask(agent_vago, pregunta, verbose=False)
_, msgs_bien = await ask(agent_v2, pregunta, verbose=False)
print("descripciones vagas →", [n for n, _ in trayectoria(msgs_vago)])
print("descripciones claras →", [n for n, _ in trayectoria(msgs_bien)])

**Qué mirar:** ¿cambió la herramienta elegida?, ¿hay pasos de más?, ¿los argumentos son razonables? Los resultados varían entre ejecuciones; lo importante es el **método**: la trayectoria te dice *dónde* se desvió el agente. **Corrección:** descripciones que digan **qué hace, cuándo usarla y cuándo NO** (como las de `web_search` y `extract_page`).

#### Falla 3 · La herramienta externa falla (clave inválida)

Simulamos que la clave de Tavily es inválida pasándole al servidor una variable de entorno distinta a la del `.env`.

In [ ]:
params_malos = server_params(env={"TAVILY_API_KEY": "tvly-clave-invalida"})
tools_malas = await load_mcp_tools(params_malos)
agent_sin_clave = create_agent(llm, tools=[consultar_menciones] + tools_malas, system_prompt=SYSTEM_PROMPT_V2)

_ = await ask(agent_sin_clave, "¿Sigue abierta la Cevichería Marea Azul de Surquillo? Verifícalo en internet.")
print("--- mcp_server.log (final) ---")
print(SERVER_LOG.read_text(encoding="utf-8")[-600:])

**Qué mirar:** la observación empieza con `ERROR de la herramienta web_search: …` y el agente debe **contarlo sin inventar** lo que no pudo verificar (regla 6 del prompt). El traceback completo está en `mcp_server.log`.

**✍️ Experimento:** en `tavily_mcp_server.py`, cambia `ToolError(...)` por `ValueError(...)` en `_explain`, vuelve a ejecutar esta celda y compara qué texto recibe el modelo. En el SDK 2.x solo los `ToolError` («fallas anticipadas») llegan con su mensaje; cualquier otra excepción se reporta como un genérico *«Error executing tool …»* y el detalle queda **solo en el log del servidor**.

#### Falla 4 · El agente inventa (falta de *grounding*)

Un prompt débil, sin reglas de fundamentación, ante una pregunta **sin datos**:

In [ ]:
agent_debil = create_agent(
    llm,
    tools=[consultar_menciones],
    system_prompt="Eres un asistente gastronómico peruano. Ayuda al usuario a elegir dónde comer.",
)
pregunta = "Recomiéndame un pollo a la brasa en Jesús María, con el nombre del lugar."
_ = await ask(agent_debil, pregunta)
_ = await ask(agent_v2, pregunta)

**Qué mirar:** el agente débil puede **completar con conocimiento general** nombres de lugares que **ningún youtuber de la base mencionó**: suena bien y es incontrolable. **Corrección:** reglas explícitas de fundamentación (*«no recomiendes nada que no salga de la herramienta»*, *«si es SIN_RESULTADOS, dilo»*) **y** un caso de evaluación que lo vigile (`sin-datos`).

#### Falla 5 · Bucles y exceso de llamadas

Un agente autónomo puede seguir llamando herramientas sin parar. Dos frenos: `recursion_limit` (pasos totales del grafo) y `ToolCallLimitMiddleware` (llamadas por herramienta).

In [ ]:
from langgraph.errors import GraphRecursionError

pregunta = "Para cada ceviche de la base, verifica en internet si su restaurante sigue abierto, uno por uno."
try:
    await ask(agent_v2, pregunta, config={"recursion_limit": 6})
except GraphRecursionError:
    print("⛔ Se alcanzó el recursion_limit: el agente fue detenido en vez de seguir indefinidamente.")

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

agent_con_limite = create_agent(
    llm,
    tools=tools_v2,
    system_prompt=SYSTEM_PROMPT_V2,
    middleware=[ToolCallLimitMiddleware(tool_name="web_search", run_limit=2)],   # máx. 2 búsquedas por ejecución
)
_, msgs = await ask(agent_con_limite, pregunta, config={"recursion_limit": 25})
print("Búsquedas web llamadas por el modelo:", sum(1 for n, _ in trayectoria(msgs) if n == "web_search"))

**Qué mirar:** con el middleware, las llamadas que exceden el límite **no se ejecutan** y el modelo recibe un aviso para que concluya con lo que tiene. En producción conviene combinar ambos frenos con un presupuesto de costo.

### 4.5 (Opcional) Trazas con LangSmith

Para ver cada paso del agente en una interfaz (prompts, tokens, tiempos, errores) activa en tu `.env`: `LANGSMITH_TRACING=true` y `LANGSMITH_API_KEY=...`. No requiere cambiar el código.

In [ ]:
if os.getenv("LANGSMITH_TRACING", "false").lower() == "true" and os.getenv("LANGSMITH_API_KEY"):
    print("LangSmith activo ✔  proyecto:", os.getenv("LANGSMITH_PROJECT", "default"))
else:
    print("LangSmith desactivado (opcional). Actívalo desde tu archivo .env.")

## Cierre · El agente Foodie completo (≈ 15 min)

Juntamos todo: **herramienta local** + **herramientas web vía tu servidor MCP** + **system prompt con reglas de fundamentación** + **memoria de conversación**. Con un `checkpointer` y un `thread_id`, el agente recuerda lo conversado dentro del mismo hilo.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

foodie = create_agent(llm, tools=tools_v2, system_prompt=SYSTEM_PROMPT_V2, checkpointer=InMemorySaver())
hilo = {"configurable": {"thread_id": "alumno-1"}}

_ = await ask(foodie, "Hola, este fin de semana quiero ceviche por Surquillo o Chorrillos. ¿Qué opciones hay?", config=hilo)

In [ ]:
_ = await ask(foodie, "Prefiero la más económica. ¿Cuál sería?", config=hilo)   # "la más económica" depende de lo anterior

In [ ]:
_ = await ask(foodie, "¿Y sigue abierta? Verifícalo en internet.", config=hilo)

### Qué construiste

| Pieza | Tipo | Idea clave |
|---|---|---|
| Extracción de menciones desde una transcripción | **Workflow** (prompt chaining + compuerta) | Pasos conocidos → código decide la ruta |
| Foodie | **Agente** (ReAct) | Pasos impredecibles → el modelo decide |
| `consultar_menciones` | Herramienta local | El docstring es un prompt |
| Servidor MCP sobre Tavily | Herramientas externas | Un estándar: la misma herramienta sirve a cualquier cliente |
| Puente MCP → LangChain | Integración | Una herramienta MCP es una función con esquema |
| Trayectorias, pruebas por capas, evaluación | Calidad | Mide el comportamiento, no solo la respuesta |

### Para seguir practicando

1. **Búsqueda semántica:** reemplaza el filtro por texto de `consultar_menciones` por *embeddings* (ahora *«suave»* no encuentra *«sin picante»*).
2. **LLM como juez** para la evaluación de la Parte 4, y repetir cada caso 5 veces para medir variabilidad.
3. **Transporte HTTP:** sirve el servidor MCP con `streamable-http` y conéctalo con un cliente por URL.
4. **Compara con el servidor MCP oficial de Tavily:** ¿qué herramientas y parámetros expone?, ¿qué decidió distinto su diseño del tuyo?
5. **Un workflow más:** *routing* para derivar «preguntas de recomendación» y «preguntas de verificación» a caminos distintos, y mide si gana en costo o en precisión frente al agente único (recuerda el criterio del post: ¿hace falta de verdad la autonomía?).

## Anexo A · (Instructor) Construir `menciones.csv` con transcripciones reales

El mismo **workflow** de la Parte 1 se aplica a **todas** las transcripciones reales para generar la tabla que usa el agente.

1. En tu máquina local (las IPs de Colab/servidores suelen estar bloqueadas por YouTube): completa `CANALES` en `collect_transcripts.py` y ejecútalo con el `.venv` activo → crea `data/transcripts/` y `data/transcripts_index.csv`.
2. Ejecuta la celda de abajo → crea `data/menciones.csv`, que el notebook usa automáticamente en lugar del demo.

**Publicación:** las transcripciones son **contenido de terceros**: por eso `data/transcripts/` está en `.gitignore`. Publica, si quieres, solo la tabla derivada (comentarios resumidos) y cita siempre canal y video.

In [ ]:
TRANSCRIPTS_DIR = Path("data/transcripts")
INDEX_CSV = Path("data/transcripts_index.csv")


def trocear(texto: str, n: int = 6000) -> list[str]:
    return [texto[i:i + n] for i in range(0, len(texto), n)]


if TRANSCRIPTS_DIR.exists() and INDEX_CSV.exists():
    indice = pd.read_csv(INDEX_CSV)
    partes = []
    for fila in indice.itertuples():
        archivo = TRANSCRIPTS_DIR / f"{fila.video_id}.txt"
        if not archivo.exists():
            continue
        for trozo in trocear(archivo.read_text(encoding="utf-8")):
            df_trozo, _ = workflow_extraer(trozo, fila.youtuber, fila.video_url, verbose=False)
            if not df_trozo.empty:
                partes.append(df_trozo)
    if partes:
        menciones = (pd.concat(partes, ignore_index=True)
                       .drop_duplicates(subset=["youtuber", "video_url", "restaurante", "plato"]))
        menciones.to_csv("data/menciones.csv", index=False)
        print(f"✔ data/menciones.csv generado con {len(menciones)} menciones")
    else:
        print("No se extrajo ninguna mención.")
else:
    print("Aún no hay transcripciones reales (data/transcripts/). Este anexo es opcional.")